# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"Dataset Name: {metadata.get('name', 'N/A')}\nDescription: {metadata.get('description', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their `@id` identifiers.

In [ ]:
# List available record sets and their fields
record_sets_info = dataset.metadata.to_json().get('recordSet', [])
if not record_sets_info:
    print('No record sets found in the dataset metadata.')
else:
    for i, record_set in enumerate(record_sets_info):
        print(f"[{i}] Record Set @id: {record_set['@id']}")
        # List available fields in the record set
        # Load the complete record set object
        record_set_metadata = dataset.metadata.get_by_id(record_set['@id'])
        fields = record_set_metadata.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        print("    Fields:")
        for field in fields:
            field_obj = dataset.metadata.get_by_id(field['@id']) if isinstance(field, dict) and '@id' in field else field
            field_id = field_obj['@id'] if isinstance(field_obj, dict) and '@id' in field_obj else field_obj
            field_name = field_obj.get('name', field_id) if isinstance(field_obj, dict) else field_id
            print(f"      - {field_id} (name: {field_name})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# For this dataset, many FAIR² record sets may exist; to demonstrate, use the primary record set if present.
primary_record_sets = dataset.metadata.to_json().get('recordSet', [])
record_sets_ids = [rs['@id'] for rs in primary_record_sets] if primary_record_sets else []
if not record_sets_ids:
    print('No available record sets found.')
else:
    print('Record Sets IDs:', record_sets_ids)

dataframes = {}
for record_set_id in record_sets_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded DataFrame for record set {record_set_id}, columns: {dataframes[record_set_id].columns.tolist()}")
            display(dataframes[record_set_id].head())
        else:
            print(f"No records found for record set {record_set_id}.")
    except Exception as e:
        print(f"Error loading records for {record_set_id}: {e}")

# For continuing steps, select the first available record set
if record_sets_ids:
    main_record_set_id = record_sets_ids[0]
    df = dataframes.get(main_record_set_id, pd.DataFrame())
    print("Columns in chosen record set:", df.columns.tolist())
    df.head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering, normalization, and grouping, using field `@id` references.

In [ ]:
# Set sample field `@id`s for numeric and group analysis.
# You should review the DataFrame columns to confirm available options.

if not df.empty:
    # Example: Suppose the DataFrame includes fields like 'cr:coverage', 'cr:molecular_weight', 'cr:abundance'
    print('Columns:', df.columns.tolist())
    # Use field IDs likely to exist in proteomics datasets; adjust to your dataset as needed:
    candidate_numeric_fields = [c for c in df.columns if any(x in c.lower() for x in ["molecular", "weight", "abundance", "coverage", "count"])]
    numeric_field = candidate_numeric_fields[0] if candidate_numeric_fields else df.columns[0]

    print(f"Using numeric field for analysis: {numeric_field}")

    # Filter
    threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0 # use mean as threshold if numeric
    filtered_df = df[df[numeric_field] > threshold] if pd.api.types.is_numeric_dtype(df[numeric_field]) else df.copy()

    print(f"Filtered {numeric_field} > {threshold} ({filtered_df.shape[0]} records)")

    # Normalize field (if numeric)
    if pd.api.types.is_numeric_dtype(filtered_df[numeric_field]):
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to group by a string/categorical field
    candidate_group_fields = [c for c in df.columns if c != numeric_field and pd.api.types.is_string_dtype(df[c])]
    if candidate_group_fields:
        group_field = candidate_group_fields[0]
        print(f'Grouping by: {group_field}')
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(grouped_df.head())
    else:
        group_field = None
else:
    print('No data available for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields using the extracted DataFrame.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not df.empty:
    # Distribution plot for numeric field
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot per group if group_field is available
    if 'group_field' in locals() and group_field and pd.api.types.is_string_dtype(df[group_field]):
        plt.figure(figsize=(12, 5))
        sns.boxplot(y=df[numeric_field], x=df[group_field])
        plt.title(f"{numeric_field} per {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No data available for visualization.')

## 6. Conclusion
This notebook loaded the FAIR² protein mass spectrometry dataset from a Croissant schema, explored available record sets and fields via their `@id`, extracted records into DataFrames, and applied basic EDA and visualizations. For more advanced analysis, cross-reference the schema documentation to select key analytical fields and tailor your workflow toward your scientific or bioinformatics use case.